# 25 - FrozenLake DQN with Residual Reasoning

Use the same fixed FrozenLake replay and shared Mouse transformer as example 23,
but replace vocabulary generation with continuous residual updates:

$$u_r = \operatorname{RMSNorm}(\operatorname{stopgrad}(z_r)),\qquad
v_r = T_\theta(c_1,\ldots,c_m,u_0,\ldots,u_r)_{m+r},\qquad
z_{r+1} = z_r + v_r.$$

Start from the tokenizer-defined generation token embedding $z_0$, perform
updates until a learned stop head fires (at most `MAX_REASONING_STEPS`), and use a
learned linear projection $Q=\operatorname{Proj}(z_R)$ for the four action values.
A separate stop head makes a deterministic decision after each update: stop when
its logit is nonnegative. Training minimizes the expected DQN squared error over
possible stopping points plus `REASONING_STEP_COST` per extra update. It evaluates
the full capped trajectory without gradients, then evaluates the generated batch once. Generation takes
the first hard stop; it never samples the stop probabilities.

Each residual state's **input to normalization** is detached. Context embeddings
remain differentiable and train through the transformer's attention to context.
The normalization scale and the entire residual sum also receive gradients.
The starting token embedding receives gradients
through the solid residual path. Each appended state occupies its own causal token
position. Existing backbone RoPE provides positional information; we deliberately
omit explicit time embeddings. RoPE is positional conditioning, not an exact
replacement for a general time embedding.

**One differentiable transformer forward:** first generate complete token sequences without an
autograd graph, then pack differentiable context embeddings and detached state inputs into
one causal forward. Reconstruct $z_R=z_0+\sum_r v_r$ from that forward and call
`loss.backward()` once. This avoids backpropagation through the rollout's feedback
edges, but still requires sequential no-grad generation before training. Gradient
checkpointing may recompute transformer layers during backward.

Install Mouse Core with its `examples` extra and use CUDA for training.
Both generation backends use FlexAttention; CPU supports inference only.
Both reasoners use the shared generation scheduler; continuous mode reuses a paged KV cache and immediately fills finished slots.
This is an executable architecture/training example, not evidence that a short
run solves FrozenLake or that more residual steps improve performance.
This notebook only loads recorded episodes, generates reasoning, trains, and saves.
Environment collection and evaluation are separate workflows.

## Setup

Imports and configuration. Edit these settings before running the cells below.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Literal, cast

import torch
from huggingface_hub import HfApi
from tensordict import TensorDict
from torch import nn
from transformers import AutoTokenizer

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    DataLoader,
    EnvironmentProtocol,
    Tokenizer,
    compose,
    environment_decisions,
    load_stores_from_hub,
    save_tokenizer,
)
from mouse_core.models import (
    Backbone,
    BaseBackbone,
    Embedding,
    Model,
    MultiEmbedding,
    ResidualBatch,
    ResidualReasoner,
    save_model,
)
from mouse_core.models.backbone.continuous_decode import ContinuousDecodeSession
from mouse_core.objectives import DqnObjective, boundary_discount

# Repositories
DATASET_ID = "mouse-example-frozenlake-reasoning-dataset"
MODEL_ID = "mouse-example-residual-dqn"

# Environment
IS_SLIPPERY = False
MAX_EPISODE_STEPS = 16

# Model
PRETRAINED = "Qwen/Qwen3-0.6B"

# Tokenizer
GENERATION_START_TOKEN = "\n"  # Existing single-token newline; no vocabulary expansion.

# Reasoning
MAX_REASONING_STEPS = 16
REASONING_STEP_COST = 0.01
MAX_ACTIVE_REQUESTS = 8
GENERATION_BACKEND: Literal["uncached", "continuous"] = "continuous"

# Training
PREFETCH = 4
TOKEN_BUDGET = 2048
NUM_WORKERS = 0
NUM_CYCLES = 1
TRAIN_STEPS = 10

# Objectives
GAMMA = 0.99

# Optimizer
LR_EMBEDDING = 1e-5
LR_BACKBONE = 1e-5
LR_HEADS = 1e-5
MAX_GRAD_NORM = 1.0

# Target updates
POLYAK_TAU_EMBEDDING = 0.1
POLYAK_TAU_BACKBONE = 0.1
POLYAK_TAU_VALUE = 0.001
# delayed stop head
POLYAK_TAU_REASONING_HEAD = 0.001

# Random seeds
SEED = 7

# Runtime options
USE_KERNELS = False
GRADIENT_CHECKPOINTING = True

## Runtime


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook trains with FlexAttention and requires CUDA; CPU supports inference only."
    )
device = torch.device("cuda")
torch.manual_seed(SEED)

## Load Data

This is offline RL. Load existing Mouse event stores; this notebook never creates
or steps an environment. Set `DATASET_ID` to a previously recorded dataset of
standalone 4x4 `FrozenLake-v1` episodes, with actions 0=left, 1=down, 2=right, 3=up.
Set `IS_SLIPPERY` and `MAX_EPISODE_STEPS` to match that dataset. The dataset ID is
an example name to replace with your own source. Use the collector's
coordinator/agent/environment event schema with `_episode_start`, `_terminated`,
and `_truncated`; repeated procedural-map tasks require their own matching prompt.

Only complete episodes are sampled. Arrival rows hold the recorded previous
action and reward. Terminal and time-limit rows have no value readout. Generating
reasoning adds model computation over these recorded histories, not interactions.
Evaluate saved checkpoints separately in [15 — Inference](15_inference.ipynb).


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train")

## Tokenizer

Render each observed arrival as a prompt event. The same tokenizer defines text
inputs, the generation-start marker, and objective labels. The initial event uses
the chat template; later events append the incoming action, reward, and observation.
Terminal observations retain their labels but have no prediction marker.

In [ ]:
def task_instructions() -> str:
    """Save the exact inference prompt alongside the checkpoint."""
    return (
        "Estimate action values in FrozenLake: a 4 by 4 grid, with states numbered row-major 0 to 15. "
        "Actions: 0=left, 1=down, 2=right, 3=up. The map is hidden. "
        "The goal rewards 1; a hole ends the episode with reward 0. "
        f"The episode ends after at most {MAX_EPISODE_STEPS} steps. "
        f"Slippery transitions: {IS_SLIPPERY}. "
        "Estimate values from the observed history.\nHistory:\n"
    )


def format_observation(*, observation, action, reward, requested):
    """Render one arrival; the tokenizer adds its action-request marker."""
    if action is None:
        text = text_tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": task_instructions() + f"State {observation}.\n",
                }
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    else:
        text = f"Action {action}, reward {reward:g}, state {observation}.\n"
    return text + ("Value:" if requested else "")


def format_events(events):
    """Convert sampled source events to prompt events; the loader owns packing."""
    protocol = EnvironmentProtocol(
        episode_start="_episode_start", terminated="_terminated", truncated="_truncated"
    )
    formatted = []
    for row in environment_decisions(events, protocol=protocol):
        observation, action = row["observation"], row.get("action")
        if row.get("inner_terminated") or row.get("inner_truncated"):
            raise ValueError(
                "Use standalone 4x4 FrozenLake episodes, without inner repeats"
            )
        if not 0 <= observation < 16 or (action is not None and not 0 <= action < 4):
            raise ValueError(
                "This example expects 4x4 FrozenLake observations and four actions"
            )
        reward = float(row.get("reward", 0.0))
        terminated = bool(row.get("_terminated", False))
        truncated = bool(row.get("_truncated", False))
        requested = bool(row.get("action_requested", False))
        if not (terminated or truncated) and not requested:
            raise ValueError("Each nonterminal observation must request an action")
        payload = {
            "prompt": format_observation(
                observation=int(observation),
                action=action,
                reward=reward,
                requested=requested,
            ),
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
        }
        if action is not None:
            payload["action"] = int(action)
        if requested:
            payload["action_requested"] = True
        formatted.append(
            {
                "provider": "environment",
                "episode_index": row["episode_index"],
                "step_index": row["step_index"],
                "payload": payload,
            }
        )
    return formatted


text_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED)
tokenizer = Tokenizer(
    tokenizers={"text": text_tokenizer},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "prompt",
            "format": "{prompt}",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": GENERATION_START_TOKEN,
            "head_output": True,
            "autoregressive": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "truncated",
            "missing_value": False,
        },
    ],
)
# Fail before model allocation if the configured marker is not a single token.
tokenizer({"provider": "environment", "payload": {"action_requested": True}})

## Replay

Use the standard `DataLoader` to sample complete recorded episodes. The transform
formats source events and calls `tokenizer.encode`; the loader packs tokens,
objective labels, sequence IDs, and prediction-to-step mappings. `TOKEN_BUDGET`
limits packed context tokens; generated reasoning has its own cap.

No environment interactions occur here. Each optimizer update consumes
`loader.next_batch()`, generates fresh reasoning at every action request, then trains.

In [ ]:
train_transform = compose(stages=(format_events, tokenizer.encode))


def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Exclude incomplete prefixes and unfinished episode tails."""
    first, last = events[0], events[-1]
    return (
        first["provider"] == "coordinator"
        and first["step_index"] == 0
        and first["payload"].get("_episode_start") is True
        and last["provider"] == "environment"
        and (
            last["payload"].get("_terminated") is True
            or last["payload"].get("_truncated") is True
        )
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    transform=train_transform,
    prefetch=PREFETCH,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

## Build Encoder and Heads

`ResidualReasoner` owns the input RMSNorm and stop head. The supplied embedding and
backbone are the same modules used for every residual update and the packed
replay. A linear value head implements the diagram's final projection.

The training loop uses eval mode while optimizing, disabling dropout without
disabling gradients. It keeps this mode through backward so gradient-checkpoint
recomputation also matches the generated trajectory. The delayed DQN model generates **its own**
residual states with its own parameters; it does not reuse online latent states.


In [ ]:
class ResidualDqn(Model):
    def __init__(
        self,
        *,
        embedding: MultiEmbedding,
        backbone: BaseBackbone,
        value_head: nn.Module,
        max_steps: int,
    ) -> None:
        super().__init__()
        # Both context and starting token embeddings train during forward.
        self.embedding = embedding
        self.backbone = backbone
        self.value_head = value_head
        self.reasoner = ResidualReasoner(
            hidden_dim=backbone.hidden_dim, max_steps=max_steps
        )

    @torch.no_grad()
    def generate(
        self,
        tokens: TensorDict,
        *,
        backend: Literal["uncached", "continuous"] = "continuous",
        max_active_requests: int = 8,
        training_candidates: bool = False,
        cache: ContinuousDecodeSession | None = None,
    ) -> ResidualBatch:
        return self.reasoner.generate(
            tokens,
            embedding=self.embedding,
            backbone=self.backbone,
            backend=backend,
            max_active_requests=max_active_requests,
            stop_early=not training_candidates,
            cache=cache,
        )

    def forward(self, generated: ResidualBatch) -> dict[str, torch.Tensor]:
        states, logits = self.reasoner(
            generated,
            embedding=self.embedding,
            backbone=self.backbone,
        )
        q_steps = self.value_head(states)
        rows = torch.arange(len(states), device=states.device)
        return dict(
            q=q_steps[rows, generated.history_lengths.to(states.device) - 1],
            q_steps=q_steps,
            states=states,
            stop_logits=logits,
        )


embedding = MultiEmbedding(embeddings={"text": Embedding(pretrained=PRETRAINED)})
backbone = Backbone(pretrained=PRETRAINED, dtype=torch.float32)
backbone.gradient_checkpointing = GRADIENT_CHECKPOINTING
value_head = nn.Linear(backbone.hidden_dim, 4)
online = ResidualDqn(
    embedding=embedding,
    backbone=backbone,
    value_head=value_head,
    max_steps=MAX_REASONING_STEPS,
).to(device)
target = online.copy().requires_grad_(False).eval()
if USE_KERNELS:
    backbone.enable_kernels()
    cast(Backbone, target.backbone).enable_kernels()

## Train

Generate a fresh episode trajectory without gradients, using the deterministic
stop head. Evaluate candidate stopping points up to `MAX_REASONING_STEPS` for each
decision against that same retained episode history. Candidate states beyond the
actual stop never become context for later decisions.

Replay the candidates in one differentiable transformer forward. Minimize the
expected DQN error plus `REASONING_STEP_COST` per extra update, using differentiable
first-stop probabilities. The delayed model generates its own hard-stopped episode
trajectory for the DQN targets. The residual sum carries the loss to each update;
feedback states are detached before normalization. Context embeddings and causal
attention remain differentiable.

Inference uses `model.generate`, stops at the first nonnegative stop logit or the
hard cap, and projects that final state to action values. All reasoning tokens
remain in the episode cache until the episode ends. `MAX_ACTIVE_REQUESTS` bounds
concurrent generation requests. `TOKEN_BUDGET` and episode/reasoning lengths
control the size of the differentiable replay.


In [ ]:
optimizer = AdamW(
    params=online.parameter_groups(
        lr={
            "embedding": LR_EMBEDDING,
            "backbone": LR_BACKBONE,
            "value_head": LR_HEADS,
            "reasoner.norm": LR_BACKBONE,
            "reasoner.stop_head": LR_HEADS,
        }
    ),
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-8,
)

objective = DqnObjective(
    reward=None,
    value=None,
    gate=None,
    temperature=0.0,
    discount=boundary_discount(
        gamma_step=GAMMA,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Sample recorded episodes, generate reasoning, and run optimizer steps."""
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    if not 0 < MAX_GRAD_NORM < float("inf"):
        raise ValueError("Invalid max_grad_norm")
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        tokens, objective_data = loader.next_batch()
        tokens = tokens.to(device)
        objective_data = objective_data.to(device)
        sequence_ids = objective_data["sequence_ids"]
        # Eval mode disables stochastic layers; gradients and optimizer updates remain enabled.
        # Keep it through backward so checkpoint recomputation uses the same mode.
        online.eval()
        target.eval()
        generated = online.generate(
            tokens,
            backend=GENERATION_BACKEND,
            max_active_requests=MAX_ACTIVE_REQUESTS,
            training_candidates=True,
        )
        # Both forward calls consume the same complete generated token sequences.
        lengths = generated.history_lengths
        out = online(generated)
        with torch.no_grad():
            delayed_q = target(generated)["q"]
        states, logits = out["states"], out["stop_logits"]
        q_steps, q = out["q_steps"], out["q"]
        _, objective_metrics = objective(
            objective_data=objective_data,
            sequence_ids=sequence_ids,
            step_ids=objective_data["step_ids"],
            predictions=q,
            backup_value_predictions=delayed_q,
            backup_action_predictions=delayed_q,
        )
        step_ids = objective_data["step_ids"]
        next_steps = (step_ids + 1).clamp_max(len(objective_data["action"]) - 1)
        actions = objective_data["action"][next_steps].nan_to_num().long()
        selected = q_steps.gather(
            -1, actions[:, None, None].expand(-1, online.reasoner.max_steps, 1)
        ).squeeze(-1)
        weights, backup = (
            objective_metrics["backup_weight"],
            objective_metrics["backup"],
        )
        if not isinstance(weights, torch.Tensor) or not isinstance(
            backup, torch.Tensor
        ):
            raise TypeError("DQN backup values and weights must be tensors")
        errors = (
            (selected - backup[:, None]).square().masked_fill(weights[:, None] <= 0, 0)
        )
        halting = online.reasoner.halting_loss(
            errors, logits, row_weights=weights, step_cost=REASONING_STEP_COST
        )
        loss = halting["loss"]
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        norm = nn.utils.clip_grad_norm_(
            online.parameters(), MAX_GRAD_NORM, error_if_nonfinite=True
        )
        optimizer.step()
        polyak_update(
            target=target.embedding,
            source=online.embedding,
            tau=POLYAK_TAU_EMBEDDING,
        )
        polyak_update(
            target=target.backbone,
            source=online.backbone,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(
            target=target.reasoner.norm,
            source=online.reasoner.norm,
            tau=POLYAK_TAU_BACKBONE,
        )
        polyak_update(
            target=target.value_head,
            source=online.value_head,
            tau=POLYAK_TAU_VALUE,
        )
        polyak_update(
            target=target.reasoner.stop_head,
            source=online.reasoner.stop_head,
            tau=POLYAK_TAU_REASONING_HEAD,
        )
        metrics = {
            "loss": float(loss.detach()),
            "dqn_loss": float(halting["task_loss"].detach()),
            "reasoning_cost": float(halting["reasoning_cost"].detach()),
            "expected_reasoning_steps": float(halting["expected_steps"].detach()),
            "grad_norm": float(norm),
            "residual_rms": float(
                states.detach()[torch.arange(len(states), device=device), lengths - 1]
                .square()
                .mean()
                .sqrt()
            ),
            "reasoning_steps": float(lengths.float().mean()),
        }
    assert loss is not None
    return (loss.detach(), metrics)

## Run

Train on fixed replay, regenerating residual trajectories for each update.
Reasoning length is learned, with `MAX_REASONING_STEPS` as the per-action hard cap.
If the stop head never fires, the last state at the cap supplies the action values.


In [ ]:
try:
    for cycle in range(NUM_CYCLES):
        loss, metrics = run_train(num_steps=TRAIN_STEPS)
        print(f"cycle={cycle} train  loss={loss.item():.4f}  {metrics}")
finally:
    loader.close()

## Save to Hugging Face

Run this cell to upload the online weights, tokenizer files and inference settings
together to `MODEL_ID` in your authenticated Hugging Face account. Temporary files
are only used to assemble the upload. Optimizer and delayed-network state are not
included. Evaluate the uploaded repository in the matching reasoning section of
[15 — Inference](15_inference.ipynb).


In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory)
    save_model(model=online, path=output)
    save_tokenizer(tokenizer=tokenizer, path=output / "tokenizer")
    text_tokenizer.save_pretrained(output / "text_tokenizer")
    (output / "residual_settings.json").write_text(
        json.dumps(
            {
                "pretrained": PRETRAINED,
                "environment": {
                    "id": "FrozenLake-v1",
                    "map_name": "4x4",
                    "is_slippery": IS_SLIPPERY,
                    "max_episode_steps": MAX_EPISODE_STEPS,
                },
                "prompt_prefix": task_instructions(),
                "max_steps": online.reasoner.max_steps,
                "reasoning_step_cost": REASONING_STEP_COST,
                "stop_rule": "first_nonnegative_logit",
                "initial_state": "generation_token_embedding",
                "generation_start_token": GENERATION_START_TOKEN,
                "input_norm": "RMSNorm",
                "norm_eps": online.reasoner.norm.eps,
                "explicit_time_embeddings": False,
                "num_actions": 4,
            },
            indent=2,
        )
        + "\n"
    )
    api = HfApi()
    repository = api.create_repo(
        repo_id=MODEL_ID, repo_type="model", private=False, exist_ok=True
    )
    api.upload_folder(
        repo_id=repository.repo_id,
        repo_type="model",
        folder_path=directory,
        commit_message="Upload reasoning model, tokenizers and inference settings",
    )
print(f"Uploaded checkpoint bundle to {repository}")